# Sentiment Analysis using Gemini, Llama3, and OpenAI
## EMC Comments Analysis
### 003 - Data Pre-Prep, Letter Parser, 1 to N Comments

Read in curated Letters from RAW CARA data and sub-parse into additional comments.

Presentation provided by Power BI.

### TODO
+ Are verbotten phrases being removed?
+ Added JSON_ERROR to try/catch
+ Handle prompt engineering with library, use prompt templates.
+ https://towardsdatascience.com/document-topic-extraction-with-large-language-models-llm-and-the-latent-dirichlet-allocation-e4697e4dae87

### Version History
+ v0.1 - General access, no cleansing of data, df.apply() for OpenAI API call.  Gaps in data output.
+ v0.2 - Same dataset (Sonya Sachedeva), robust cleansing, lemmatizing, and stemming.  Summary of summary for token limit solved.
+ v0.3 - Added prompt defense, PII defense, df.apply() with defensive method, dropping lemmatizing/stemming.  Added libraries such as commonregex, spacy, and transformers.
+ v0.4 - Broke into data processing versus data prepping functions.
+ v0.5 - Moved core code into "main" to support multi-processing in future, backed up original data to 'Letter Text_ORIGINAL', explored multi-processing and GPU utilization.
+ v0.6 - New dataset to process direct from CARA extract.  Sonya Sachedeva's data inputs processed with v0.5 which has been tagged.
  

In [ ]:
# -*- coding: utf-8 -*-

### Environment Validation

Using GCP or Azure read in arrays representing minimal library requirements (which might not be present in a Google Colab environment) and install / load the libraries as required.  Additional imports for standard libraries and tailored content to follow.

In [ ]:
###########################################
#- Minimal imports to start
###########################################
try:
    import sys
    import subprocess
    import importlib.util
    import atexit
except ImportError as e:
    print("There was a problem importing the most basic libraries necessary for this code.")
    print(repr(e))
    raise SystemExit("Stop right there!")

###########################################
#- Final Exit Routine
###########################################
@atexit.register
def goodbye():
    print("GOODBYE")

###########################################
#- Cloud Environment Setup (Priming)
###########################################
# variables establishing environments
ENV_GCP=0
ENV_AZURE=1
user_input=-1
environments=["GCP", "Azure"]
    
#prompt user for environment before continuing
user_input = 1
while True:
  try:
     if user_input > -1:
         break;
     user_input = int(input("Select the environment you're running: (0) GCP (1) Azure"))     
     if user_input > 1:
         print("Not a valid choice, please try again.")
         continue;
  except ValueError:
     print("Not a valid choice, please try again.")
     continue
  else:
     print(f"Environment selected is: {environments[user_input]}")
     break 
        
############################################
#- Import a custom library, in this case a fairly useful logging framework
############################################
from pathlib import Path
debug_lib_location = Path("./")
sys.path.append(str(debug_lib_location))
import debug

libraries=["transformers", "langchain", "backoff",
           "alive-progress", "tqdm", "pyspellchecker", "wordcloud", "langchain", "icecream", "numba", 
           "fitz","dataclasses", "commonregex", "transformers", "spacy", "PyMuPDF", "PyPDF2", "pdfminer", 
           "pdfplumber","pdf2image","pytesseract"]    
debug.msg_info(f"Validating environment for the following pip packages: {libraries}")

#load environment for non-generative libraries
try:
    for library in libraries:
      if library == "Pillow":
        spec = importlib.util.find_spec("PIL")
      else:
        spec = importlib.util.find_spec(library)
      if spec is None:
        print("...installing library " + library)
        subprocess.run(["pip", "install" , library, "--quiet"])
      else:
        print("...library " + library + " already installed.")
except (subprocess.CalledProcessError, Exception) as e:
    print("Error: Failed to install required packages, your code might not run properly.")
    print(repr(e))

#load environment specific libraries for generative AI.
try:    
    if environments[user_input]=="GCP":
      subprocess.run(["pip", "install" , "--upgrade", "google-cloud-aiplatform", "--quiet"])
      gcp_libraries=["google-generativeai", "google-cloud-secret-manager"]
      for library in gcp_libraries:
        spec = importlib.util.find_spec(library)
        if spec is None:
          print("...installing library " + library)
          try:
              subprocess.run(["pip", "install" , library, "--quiet"])
          except (subprocess.CalledProcessError, Exception) as e:
              print("Error: Failed to install required packages, your code might not run properly.")
              print(repr(e))
        else:
          print("...library " + library + " already installed.")
    
        from google.cloud import aiplatform
        import vertexai.preview
        from google.cloud import secretmanager
    elif environments[user_input]=="Azure":
      azure_libraries=["openai", ]
      for library in azure_libraries:
        spec = importlib.util.find_spec(library)
        if spec is None:
          print("...installing library " + library)
          try:
              subprocess.run(["pip", "install" , library, "--quiet"])
          except (subprocess.CalledProcessError, Exception) as e:
              print("Error: Failed to install required packages, your code might not run properly.")
              print(repr(e))
        else:
          print("...library " + library + " already installed.")
    else:
        print("There was a problem processing your request.  Only numeric input of 0 or 1 is allowed.")
        print("Continued operations is not possible without the proper installed tools.")
        raise SystemExit("Stop right there!")
except Exception as e:
    print("There was a problem processing library installs for Generative AI libraries")
    print(repr(e))
    raise SystemExit("Stop right there!")

debug.msg_debug("...dynamic environment installs complete.")

## Includes and Libraries

In [ ]:
debug.msg_info("Library imports")    
############################################
# INCLUDES
############################################

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# a set of libraries that perhaps should always be in Python source
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...core libraries.")
import os
import datetime
import gc
import socket
import sys
import getopt
import inspect
import traceback
import warnings
import json
import pickle
from pathlib import Path
import itertools
import datetime
import re
import shutil
import string
from io import StringIO
import tqdm


import io
import math
import textwrap
import random
import glob
import time
from time import perf_counter
import subprocess
import backoff

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Function Profiling
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import cProfile
import pstats
import io
from pstats import SortKey

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Data Science Libraries
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...classic data science libraries.")

#optimization routines
from numba import jit
import numpy as np
import scipy as sp
#from sklearn.linear_model import LinearRegression


# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Additional libraries for this work
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...application specific libraries.")
import math
from base64 import b64decode
from IPython.display import Image
import requests
from bs4 import BeautifulSoup                 #used to parse the text
from wordcloud import WordCloud, STOPWORDS    #custom library specifically designed to make word clouds
from spellchecker import SpellChecker
import fitz
#to handle strange characters
from unidecode import unidecode 

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Graphics
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...graphics.")
#import PIL
from PIL import Image
import PIL.ImageOps
import matplotlib as matplt
import matplotlib.pyplot as plt

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# progress bar
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...progress bars.")
from alive_progress import alive_bar
#from alive_progress.styles import showtime, Show
from tqdm.notebook import trange, tqdm
#from tqdm import trange, tqdm

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#- PII libraries (regular expressions)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...regular expressions for PII and transformers for prompt injection defense.")
from commonregex import CommonRegex
from commonregex import email
from commonregex import time
from commonregex import credit_card
from commonregex import ip
from commonregex import ipv6
from commonregex import link
from commonregex import phone
from commonregex import street_address
from commonregex import btc_address

debug.msg_debug("...spacy (pii defense).")
import spacy
from spacy.language import Language
from spacy.tokens import Doc

debug.msg_debug("...hugging face model support.")
#injection defense
from transformers import pipeline

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#- Tensorflow AI/ML libraries (seek to use GPU's)
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#load first
try:
    debug.msg_debug("...TensorRT")    
    import tensorrt
    assert tensorrt.Builder(tensorrt.Logger())
except ImportError as ie:
    debug.msg_warning("Failed to import tensorrt, this might be a problem if trying for enhanced processing.")
    debug.msg_warning(f"...{repr(ie)}")
    pass

try:
    #load second
    debug.msg_debug("...TensorFlow")        
    import tensorflow as tf
except ImportError as ie:
    debug.msg_warning("Failed to import tensorflow, might not have a GPU or the proper environment loaded")
    debug.msg_warning(f"...{repr(ie)}")
    pass

try:
    debug.msg_debug("...CUDF")    
    import cudf
except ImportError as ie:
    debug.msg_warning("Failed to import cudf, likely don't have a GPU")
    debug.msg_warning(f"...{repr(ie)}")
    pass

try:
    debug.msg_debug("...Torch")    
    import torch
except ImportError as ie:
    debug.msg_warning("Failed to import torch, likely don't have a GPU or access to that library.")
    debug.msg_warning(f"...{repr(ie)}")
    pass

import pandas as pd
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#- NLTK required resources
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
debug.msg_debug("...natural language processing.")
import nltk
from nltk.stem import PorterStemmer  # A word stemmer based on the Porter stemming algorithm.  Porter, M. "An algorithm for suffix stripping." Program 14.3 (1980): 130-137.
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.tree import tree
#from nltk.book import *
from nltk import FreqDist
from nltk import sent_tokenize, word_tokenize
from nltk.corpus import stopwords    

nltk.download('punkt')
nltk.download("words")
nltk.download("stopwords")
#nltk.download('averaged_perceptron_tagger')      #looks like you have to download select neural layers for specific functions, head to read the erorr output to learn this.


## Functions

### Numpy / Pandas Configuration Settings

In [ ]:
def set_library_configuration() -> None:
    
    ############################################
    #- JUPYTER NOTEBOOK OUTPUT CONTROL / FORMATTING
    ############################################
    #pandas set floating point to 4 places to things don't run loose
    debug.msg_info("Setting Pandas and Numpy library options.")    
    pd.set_option('display.max_colwidth', 10) # None if you want to view the full json blob in the printed dataframe, use this
    pd.options.display.float_format = '{:,.4f}'.format
    np.set_printoptions(precision=4)

## Functions

### Custom Exception Display

In [ ]:
## Manages exception output.
#  @param   (Exception)             - Exception to expound upon
#  @returns (None)                  - None
def process_exception(inc_exception) -> None:
    print(f"{BOLD_START}(Exception encountered):{BOLD_END} {type(inc_exception).__name__}")
    print(f"Details: {str(inc_exception)}")
    print("Traceback:")
    traceback.print_exc()

In [ ]:
def profile_function(func):
    def wrapper(*args, **kwargs):
        pr = cProfile.Profile()
        pr.enable()
        result = func(*args, **kwargs)
        pr.disable()
        s = io.StringIO()
        sortby = SortKey.CUMULATIVE
        ps = pstats.Stats(pr, stream=s).sort_stats(sortby)
        ps.print_stats()
        print(s.getvalue())
        return result
    return wrapper

### Library Versioning Display

In [ ]:
## Outputs library version history of effort.
#
#  @returns (None)                  - None
def lib_diagnostics() -> None:

    import pkg_resources
    
    debug.msg_info(f"Entering {__name__} {inspect.stack()[0][3]}") 
    
    package_name_length=40
    package_version_length=20

    # Get installed packages
    the_packages=["cupy", "jupyter-core", "langchain", "langchain-core", "nltk", "numba", "numpy", "pandas", "pydantic", "pyspellchecker", "spacy", "scipy", "scikit-learn", "seaborn", "usaddress", "xarray",]
    the_packages.sort()
    
    installed_dict = {pkg.key: pkg.version for pkg in pkg_resources.working_set}
    installed=list(installed_dict.keys())
    installed.sort()
    
    #for package_idx, package_name in enumerate(installed):
    for idx, name in enumerate(installed):
         if name in the_packages:
             installed_version = installed_dict[name]
             print(f"{name:<40}#: {str(pkg_resources.parse_version(installed_version)):<20}")
   
    try:
        print(f"{'TensorFlow version':<40}#: {str(tf.__version__):<20}")
        print(f"{'     gpu.count:':<40}#: {str(len(tf.config.experimental.list_physical_devices('GPU')))}")
        print(f"{'     cpu.count:':<40}#: {str(len(tf.config.experimental.list_physical_devices('CPU')))}")
    except Exception as e:
        pass

    try:
        print(f"{'Torch version':<40}#: {str(torch.__version__):<20}")
        print(f"{'     GPUs available?':<40}#: {torch.cuda.is_available()}")
        print(f"{'     count':<40}#: {torch.cuda.device_count()}")
        print(f"{'     current':<40}#: {torch.cuda.current_device()}")
    except Exception as e:
        pass


    try:
      print(f"{'OpenAI Azure Version':<40}#: {str(the_openai_version):<20}")
    except Exception as e:
      pass

    print(f"{BOLD_START}List Devices{BOLD_END} #########################################")
    try:
      from tensorflow.python.client import device_lib
      print(device_lib.list_local_devices())
      print("")
    except RuntimeError as e:
      # Visible devices must be set before GPUs have been initialized
      print(str(repr(åe)))

    print(f"{BOLD_START}Devices Counts{BOLD_END} ########################################")
    try:
      print(f"Num GPUs Available: {str(len(tf.config.experimental.list_physical_devices('GPU')))}" )
      print(f"Num CPUs Available: {str(len(tf.config.experimental.list_physical_devices('CPU')))}" )
      print("")
    except RuntimeError as e:
      # Visible devices must be set before GPUs have been initialized
      print(str(repr(e)))

    print(f"{BOLD_START}Optional Enablement{BOLD_END} ####################################")
    try:
      gpus = tf.config.experimental.list_physical_devices('GPU')
    except RuntimeError as e:
      # Visible devices must be set before GPUs have been initialized
      print(str(repr(e)))

    if gpus:
      # Restrict TensorFlow to only use the first GPU
      try:
        tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print( str( str(len(gpus)) + " Physical GPUs," + str(len(logical_gpus)) + " Logical GPU") )
      except RuntimeError as e:
        # Visible devices must be set before GPUs have been initialized
        print(str(repr(e)))
      print("")
        
    debug.msg_info(f"Exiting {__name__} {inspect.stack()[0][3]}") 
    return

In [ ]:
## Main routine that executes all code, does return a data frame of data for further analysis if desired.
#
#  @param (None)
def process() -> None:

        debug.msg_info(f"Entering {__name__} {inspect.stack()[0][3]}")

        #establish data version, aligned with code
        prepended_filename="2020_"
        data_version_release="-".join([str(VERSION_NAME), str(VERSION_MAJOR), str(VERSION_MINOR), str(VERSION_RELEASE)])
        target_filename=f"./{prepended_filename}{data_version_release}"+"_EMC_Comments.bin"
        try:
            df = pickle.load(open(target_filename, "rb"))
        except (pickle.UnpicklingError, FileNotFoundError, IOError, Exception)  as e:
            debug.msg_warning("FAILED to unpickle the saved binary file, you might have corruption, investigate.")
            process_exception(e)
        except Exception as e:
            process_exception(e)
        
        #CGW FIX, TODO
        df = df.sample(n=1)

        #extract topics
        #extract keywords
        #create summary of text
        #create 1 to N comments

        """
        ####################################################################################################################
        #- Save results to datafile
        ####################################################################################################################
        debug.msg_info("Saving completed data file")
        
        #establish data version, aligned with code
        data_version_release="-".join([str(VERSION_NAME), str(VERSION_MAJOR), str(VERSION_MINOR), str(VERSION_RELEASE)])
        
        #save results of all data to binary file
        debug.msg_debug("Saving cleansed/prepped data to binary file.")
        target_filename=f"./{data_version_release}"+"_EMC_Comments.bin"
        try:
            with open(target_filename, 'wb') as file:
                pickle.dump(df, file)
            with open(target_filename, 'rb') as file:
                me = pickle.load(file)
        except (pickle.UnpicklingError, FileNotFoundError, IOError, Exception)  as e:
            debug.msg_warning("FAILED to unpickle the saved binary file, you might have corruption, investigate.")
            process_exception(e)
            
        debug.msg_debug(f"...saved and reloaded {target_filename}")
        """
    
        debug.msg_info(f"Exited {__name__} {inspect.stack()[0][3]}")

## Main

In [ ]:
if __name__ == "__main__":

    set_library_configuration()
    start_t=perf_counter()
    print("BEGIN PROGRAM")

    ############################################
    # GLOBAL CONFIGURATION
    ############################################
    #used for values outside standard ASCII, just do it, you'll need it
    ENCODING  ="utf-8"
    os.environ['PYTHONIOENCODING']=ENCODING
    #spacy requirement
    os.environ['TOKENIZERS_PARALLELISM']="false"

    debug.msg_info("Variable declaration.")    
    ############################################
    # GLOBAL VARIABLES
    ############################################
    DEBUG = 1
    DEBUG_DATA = 0
    
    # CODE CONSTRAINTS
    VERSION_NAME    = "CMTANL"
    VERSION_MAJOR   = 0
    VERSION_MINOR   = 6
    VERSION_RELEASE = 0
    
    TEXT_WIDTH=77
    BOLD_START = "\033[1m"
    BOLD_END = "\033[0;0m"
    
    ###########################################
    #- API Parameters for things like WordCloud
    ###########################################
    IMG_BACKGROUND=None                        #None without quotes or "black", "white", etc...
    IMG_FONT_SIZE_MIN=14
    IMG_WIDTH=800
    IMG_HEIGHT=600
    
    ############################################
    # APPLICATION VARIABLES
    ############################################
    SPELL_CHECK_DISTANCE=2
    MINIMUM_AI_RESPONSE=25                     #words
    MINIMUM_AI_WAIT=15                         #seconds
    os.environ["MINIMUM_AI_WAIT"] = "str(MINIMUM_AI_WAIT)"
    MINIMUM_LETTER_LENGTH=15
    SOURCE_COLUMNS_NAME=["Letter Text"]        #body of text where the actual comment is
    SOURCE_COLUMNS_IDX=[ 10 ]                   #location in data frame AFTER removal of columns
    SOURCE_COLUMN_NAME=SOURCE_COLUMNS_NAME[0]
    EVALUATION_RECORDS=10
    ERROR_PHRASE = 'Error code: 400'
    OPENAI_RESULT="ResultOPENAI"
    DATA_DIR='/projects/data/nlp/NLP_Public_Comments_20210601'
    
    
    ############################################
    # MODEL PARAMETERS
    ############################################
    #model parameters
    the_model="gpt-35-turbo-16k"
    model_temperature=0.7
    model_max_tokens=8000
    model_top_p=0.95
    model_frequency_penalty=0
    model_presence_penalty=0
    summary_token_max=150
    
    ############################################
    # PROMPT PARAMETERS
    ############################################
    PROMPT_SUMMARY_LIMIT="200"                   #number of words to generate
    PROMPT_SUMMARY_METHOD=" abstractive "        #abstractive or extractive
    
    JSON_PAYLOAD = '{ \
      "Sentiment": 0.0 \
      "SentimentConfidence": 0.0, \
      "Category": "Some Category", \
      "CitingLaw": "TRUE", \
      "BriefSummary": "Summary information about the topic.", \
      "SpecificFeedback": "TRUE", \
      "ProposedResolution":"Proposed change for improvement." \
    }'
    
    JSON_ERROR = ' { \
        "Sentiment": 100, \
        "SentimentConfidence": 100, \
        "Category": "error", \
        "CitingLaw": "error", \
        "BriefSummary": "error", \
        "SpecificFeedback": "error", \
        "ProposedResolution": "error"\
    }'
    
    EXAMPLE_JSON = '{ \
      "Sentiment": -0.5, \
      "SentimentConfidence": 0.8, \
      "Category": "Hydrology", \
      "CitingLaw": "TRUE", \
      #"Tags": ["water quality", "pollution", "regulation"], \
      "BriefSummary": "The letter expresses concerns over river pollution impacting community health and calls for stricter regulatory oversight and sustainable practices.", \
      "SpecificFeedback": "TRUE", \
      "ProposedResolution":"Stricter regulation on industry waste dumping." \
    }'
    #you could also have it propose a response that people could review/use as a starting point here
    #"SuggestedResponse":"Thank you for your thoughtful feedback regarding the dumping regulations. We share your concern about the need for robust measures to protect our environment and public health. Please know that your comments are invaluable to us and contribute significantly to our ongoing review process aimed at strengthening our regulations. We are actively working with various stakeholders, including environmental experts and community leaders, to ensure our policies effectively address these concerns while promoting sustainable practices. To stay engaged and informed about the progress and developments in this area, we encourage you to visit our website regularly and participate in upcoming public forums. Your active involvement is essential as we strive to enhance our environmental policies for the betterment of our community and future generations."
    
    PROMPT_INJECTION_MODELS=[
                             #https://python.langchain.com/v0.1/docs/guides/productionization/safety/hugging_face_prompt_injection/
                             #from optimum.onnxruntime import ORTModelForSequenceClassification
                             "protectai/deberta-v3-base-prompt-injection-v2",
                            ]
    
    ########################################
    #Define Potential "answers" from the various neural layers
    #Is the Prompt Detected?
    ########################################
    id2label = {
        'LEGIT':    False,
        'POSITIVE': False,
        'LABEL_1':  False,
        'SAFE':     False,
        
        'INJECTION':True,
        'NEGATIVE': True,
        'LABEL_0':  True,
        'UNSAFE':   True,
    }
    prompt_defense_model_chunk_size=512
    
    #python -m spacy download en
    #python -m spacy download en_core_web_sm
    #SPACY_MODEL="en_core_web_sm"
    #python -m spacy download en_core_web_lg
    #SPACY_MODEL="en_core_web_lg"
    #python -m spacy download en_core_web_trf
    SPACY_MODEL="en_core_web_trf"
    
    ############################################
    # GLOBAL CONFIGURATION
    ############################################
    os.environ['PYTHONIOENCODING']=ENCODING
    os.environ['TOKENIZERS_PARALLELISM']="false"
    
    ############################################
    #- Invocation of functions and instantiation of system needs, nltk instantiation
    ############################################
    debug.msg_debug(f"...StopWords instantiated.")
    stop_words = set(stopwords.words("english"))
    debug.msg_debug(f"...PortStemmer instantiated.")
    stemmer = PorterStemmer()
    debug.msg_debug(f"...Lemmatizer instantiated.")
    lemmatizer = WordNetLemmatizer()
    spell = SpellChecker(distance=SPELL_CHECK_DISTANCE)
    
    #setup the text wrapper
    debug.msg_debug(f"...Text Wrapper instantiated.")
    wrapper = textwrap.TextWrapper(width=TEXT_WIDTH)
    
    #show your libraries
    lib_diagnostics()
    
    ############################################
    #load the injection models for future use
    ############################################    
    the_models=[]
    try:
        debug.msg_info(f"Loading prompt injection model(s)")
        for model_idx,model_name in enumerate(PROMPT_INJECTION_MODELS):
            the_models.append(pipeline("text-classification", model=str(model_name), device=0))
            debug.msg_debug(f"...loaded {model_name}")
    except Exception as e:
        process_exception(f"ERROR detected trying to use the transformer.pipeline API to load hugging face models as follows: {str(e)}")

    ############################################
    #Core routine
    ############################################
    process()
    
    end_t=perf_counter()
    print("END PROGRAM")
    print(f"Elapsed time: {end_t - start_t}")